# Oturum 5a — Atomistik Trajektori Analizi

**Biyofizik 2026 Kursu · Doç. Dr. Mustafa Tekpınar**

Önceki oturumlarda simülasyon girdilerinin hazırlanması ele alınmıştır. Bu
not defterinde, elde edilmiş bir trajektorinin hangi ölçütlerle
değerlendirileceği incelenmektedir.

Kursta üretim simülasyonu koşulmadığından önceden hesaplanmış bir trajektori
kullanılmaktadır.


---
## 1. Yazılım kurulumu


In [ ]:
%%capture
!apt-get -qq update
!apt-get -qq install -y gromacs
!pip install -q MDAnalysis


In [ ]:
!gmx --version 2>&1 | grep -i 'GROMACS version'


---
## 2. Trajektorinin yüklenmesi

Kurs deposundaki hazır dosyalar kullanılmaktadır. Katılımcılar kendi `md.tpr`
ve `md.xtc` dosyalarını sol paneldeki dosya yöneticisi aracılığıyla da
yükleyebilirler.


In [ ]:
!git clone -q https://github.com/eygpcr/biyofizik2026-martini.git repo
!ls -la repo/05_analiz/trajektori/


In [ ]:
import os, shutil
src = 'repo/05_analiz/trajektori'
for f in os.listdir(src):
    if f.endswith(('.tpr','.xtc','.gro')):
        shutil.copy(os.path.join(src, f), '.')
!ls -lh *.tpr *.xtc 2>/dev/null || echo 'Dosyalar sol panelden yuklenmelidir'


---
## 3. Periyodik sınır koşullarının düzeltilmesi

Analiz öncesinde yapılması gereken ve sıklıkla atlanan adımdır. Periyodik sınır
koşulları nedeniyle molekül kutunun bir yüzeyinden çıkıp karşı yüzeyden
girmektedir. Düzeltme yapılmadığında hesaplanan RMSD değerleri yanıltıcı
olmaktadır.


In [ ]:
# 1) Sicramalarin kaldirilmasi
!echo 0 | gmx trjconv -s md.tpr -f md.xtc -o md_nojump.xtc -pbc nojump

# 2) Proteine gore hizalama (once hizalama grubu, ardindan cikti grubu)
!echo -e '1\n0' | gmx trjconv -s md.tpr -f md_nojump.xtc -o md_fit.xtc -fit rot+trans


---
## 4. RMSD — sistemin dengelenmesinin değerlendirilmesi

**Yorumlama.** Eğrinin bir plato değerine ulaşması sistemin dengelendiğine
işaret etmektedir. Sürekli artış gösteren bir eğri, simülasyon süresinin
yetersiz olduğunu veya yapısal bozulma bulunduğunu göstermektedir.


In [ ]:
!echo -e '4\n4' | gmx rms -s md.tpr -f md_fit.xtc -o rmsd.xvg


### Çizim fonksiyonları

GROMACS analiz araçları düz metin biçiminde `.xvg` dosyaları üretmektedir.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def xvg(path):
    """GROMACS .xvg dosyasini numpy dizisi olarak okur."""
    return np.loadtxt(path, comments=['#','@'])

def ciz(path, xlabel, ylabel, baslik, sutun=1):
    d = xvg(path)
    plt.figure(figsize=(8,4))
    plt.plot(d[:,0], d[:,sutun], lw=1.2)
    plt.xlabel(xlabel); plt.ylabel(ylabel); plt.title(baslik)
    plt.grid(alpha=.3); plt.tight_layout(); plt.show()
    return d


In [ ]:
d = ciz('rmsd.xvg', 'Zaman (ps)', 'RMSD (nm)', 'Omurga RMSD')
print(f'Son %20 icin ortalama: {d[int(len(d)*.8):,1].mean():.3f} nm')


**Tartışma sorusu.** Referans yapı olarak ilk kare mi yoksa kristal yapı mı
alınmalıdır? Bu seçim sonucu nasıl etkilemektedir? Analiz hangi zaman
aralığından itibaren yapılmalıdır?


---
## 5. RMSF — rezidü bazında dalgalanma

**Yorumlama.** İlmikler ve terminal bölgeler yüksek, düzenli ikincil yapı
öğeleri düşük değerler vermektedir. Beklenmeyen bir bölgede yüksek dalgalanma,
ya biyolojik açıdan anlamlı bir bulguya ya da kurulum hatasına işaret
etmektedir.


In [ ]:
!echo 4 | gmx rmsf -s md.tpr -f md_fit.xtc -o rmsf.xvg -res
d = ciz('rmsf.xvg', 'Rezidu numarasi', 'RMSF (nm)', 'Rezidu bazinda dalgalanma')

en_esnek = d[d[:,1].argsort()[-10:]][::-1]
print('En yuksek dalgalanma gosteren 10 rezidu:')
for r, v in en_esnek:
    print(f'  rezidu {int(r):>4}: {v:.3f} nm')


**Tartışma sorusu.** Hesaplanan RMSF profili kristal yapının B-faktörleriyle
uyum göstermekte midir?


---
## 6. Jirasyon yarıçapı — yapısal kompaktlığın izlenmesi

Ani artış, yapının açılmasına veya kurulum hatasına işaret etmektedir.


In [ ]:
!echo 4 | gmx gyrate -s md.tpr -f md_fit.xtc -o gyrate.xvg
ciz('gyrate.xvg', 'Zaman (ps)', 'Rg (nm)', 'Jirasyon yaricapi');


---
## 7. Hidrojen bağı analizi

Bu analiz Oturum 5b'de kaba-taneli modeller için karşılaştırma noktası
oluşturacaktır; Martini modelinde hidrojen atomu bulunmadığından uygulanabilir
değildir.


In [ ]:
!echo -e '1\n1' | gmx hbond -s md.tpr -f md_fit.xtc -num hbond.xvg
ciz('hbond.xvg', 'Zaman (ps)', 'Hidrojen bagi sayisi', 'Protein ici hidrojen baglari');


---
## 8. Yoğunluk profili (membran içeren sistemler)

Membran normali boyunca su, lipit baş grupları, açil zincirler ve proteinin
dağılımını vermektedir. Membran kalınlığı bu profilden belirlenmektedir.


In [ ]:
!echo 0 | gmx density -s md.tpr -f md_fit.xtc -o density.xvg -d Z -sl 100
ciz('density.xvg', 'z (nm)', 'Yogunluk (kg/m3)', 'Membran normali boyunca yogunluk');


---
## Trajektori değerlendirme ölçütleri

Yeni bir trajektori incelenirken denetlenmesi önerilen hususlar:

1. Periyodik sınır koşulları düzeltilmiş midir?
2. Enerji ve sıcaklık zaman içinde kararlı seyretmekte midir?
3. RMSD plato değerine ulaşmış mıdır; analiz hangi zaman aralığından itibaren
   yapılmalıdır?
4. Sistemde yapısal bozulma bulunmakta mıdır? Görsel denetim gereklidir.
5. Membran içeren sistemlerde lipit başına alan ve membran kalınlığı deneysel
   değerlerle uyumlu mudur?

---

## Kaynaklar

- [GROMACS analiz araçları](https://manual.gromacs.org/current/user-guide/cmdline.html#commands-by-topic)
- [GROMACS öğretim materyalleri](https://tutorials.gromacs.org/)
- [MDAnalysis](https://www.mdanalysis.org/)

**Sonraki oturum.** Aynı analizlerin kaba-taneli modellere uygulanması hâlinde
ortaya çıkan farklılıklar:
[05_analiz/martini](https://github.com/eygpcr/biyofizik2026-martini/tree/main/05_analiz/martini)
